In [ ]:
########################################
#getting system arguments
import sys
def GetArg_dataName(default="Variables"):
    """
    Safely retrieve dataName from sys.argv.
    #Run One: python Tracked_Profiles.py Variables
    #Run Two: python Tracked_Profiles.py Entrainment (also can add _DivideMassFlux)
    #Run Three: python Tracked_Profiles.py PROCESSED_Entrainment (also can add _DivideMassFlux)
    #Run Four: python Tracked_Profiles.py W_Budgets
    #Run Five: python Tracked_Profiles.py QV_Budgets
    #Run Six: python Tracked_Profiles.py TH_Budgets
    """
    # If run inside Jupyter, sys.argv will include ipykernel arguments
    if any("ipykernel_launcher" in arg for arg in sys.argv):
        print(f"Using default dataName: {default}")
        return default

    # If a user-specified argument exists, use it
    if len(sys.argv) > 1:
        out=sys.argv[1]
        print(f"Using argument dataName: {out}")
        return out

    return default

dataName = GetArg_dataName()

In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm
import copy

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=3)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
                                ModelData.Np_str, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#data manager class (for saving data)
DataManager_TrackedProfiles = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
                                ModelData.Np_str, dataType="Tracked_Profiles", dataName="Tracked_Profiles",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class, Results_InputOutput_Class, TrackedParcel_Loading_Class

In [ ]:
# IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","3_Tracked_Profiles"))
from CLASSES_TrackedProfiles import TrackedProfiles_DataLoading_CLASS

In [ ]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
recombine=True

In [ ]:
def RecombineProfiles(ModelData, DataManager, tIndicies, dataName):
    """
    Combine tracked profiles across all timesteps using the first as a template.
    """
    print(f"Recombining {ModelData.Ntime} TrackedProfiles files...\n")

    trackedProfileArrays = None

    varNames=['QV','QCQI','W','THETA_v']
    arrayNames=['profile_array_right']

    for t in tqdm(tIndicies, desc="Combining Profiles", unit="timestep"):
        if "Entrainment" in dataName and t == ModelData.Ntime-1:
            continue
        profileArraysDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData, DataManager, dataName, t)

        if t == tIndicies[0]:
            # Deep copy structure so we don’t overwrite the first timestep’s data
            trackedProfileArrays = copy.deepcopy(profileArraysDictionary)
            ProfileArraysDictionary_2D = InitializeProfileArraysDictionary_2D(profileArraysDictionary,
                                                                              varNames,arrayNames)
            continue

        # Add all later times to the initial one
        for key1 in ProfileArraysDictionary_2D:
            for key2 in ProfileArraysDictionary_2D[key1]:
                for varName in ProfileArraysDictionary_2D[key1][key2]:
                    for arrayName in arrayNames:
                        ProfileArraysDictionary_2D[key1][key2][varName][arrayName][t, :] += (
                            profileArraysDictionary[key1][key2][varName][arrayName][:, 0]
                        )
                        ProfileArraysDictionary_2D[key1][key2][varName][f"{arrayName}_count"][t, :] += (
                            profileArraysDictionary[key1][key2][varName][arrayName][:, 1]
                        )
    return ProfileArraysDictionary_2D

def InitializeProfileArraysDictionary_2D(profileArraysDictionary, 
                                         varNames,arrayNames):
    newDict = {}
    for Key1 in profileArraysDictionary:
        newDict[Key1] = {}
        for Key2 in profileArraysDictionary[Key1]:
            newDict[Key1][Key2] = {}
            for varName in varNames:
                newDict[Key1][Key2][varName] = {}
                for arrayName in arrayNames:
                    newDict[Key1][Key2][varName][arrayName] = np.zeros((Nt,Nz))
                    newDict[Key1][Key2][varName][f"{arrayName}_count"] = np.zeros((Nt,Nz))

    return newDict

Nt=ModelData.Ntime; Nz=ModelData.Nzh

In [ ]:
if recombine==True:
    dataName='Variables'
    
    ProfileArraysDictionary_2D = RecombineProfiles(ModelData, DataManager_TrackedProfiles,tIndicies=np.arange(ModelData.Ntime),dataName=dataName)
    #SaveHere

In [ ]:
#########################################
#PLOTTING FUNCTIONS

In [ ]:
def NanDivide(a, b):
    return np.divide(a, b, where=b!=0, out=np.full_like(a, np.nan, dtype=float))
def NanSubtract(a, b):
    return np.subtract(a, b)

cmapDictionary={"QV": "turbo",
                "QCQI": "turbo",
                "W": "RdBu_r",
                "THETA_v": "turbo"}

In [ ]:
def MakePlot(parcelType, parcelDepth, varName,
             plotType='contour', nLevels=41, axis=None):

    if axis is None:
        axis = plt.gca()

    profile = ProfileArraysDictionary_2D[parcelType][parcelDepth][varName]['profile_array_right']
    count = ProfileArraysDictionary_2D[parcelType][parcelDepth][varName]['profile_array_right_count']
    mean_profile = NanDivide(profile, count)
    
    times = np.arange(Nt) * ModelData.dt / 3600 + 6
    z = ModelData.zh
    
    multiplier = 1000 if "Q" in varName else 1
    extend = 'both' if "W" in varName else 'max'
    
    data = multiplier * mean_profile.T
    cmap = cmapDictionary[varName]

    if varName == "W":
        vmax = np.nanpercentile(data,95)
        vmin = -vmax
    else:
        vmin = np.nanmin(data)
        vmax = np.nanmax(data)
    if varName == "QCQI":
        vmin = 1e-6
    levels = np.linspace(vmin, vmax, nLevels)

    if plotType == 'contour':   
        plot = axis.contourf(times, z, data, cmap=cmap, levels=levels, extend=extend)
    
    elif plotType == 'pcolormesh':
        norm = mcolors.BoundaryNorm(levels, ncolors=plt.cm.get_cmap(cmap).N, extend=extend)
        plot = axis.pcolormesh(times, z, data, cmap=cmap, norm=norm, shading='auto')
    return plot

def MakeDifferencePlot(parcelTypes, parcelDepth, varName,
                       plotType='contour', nLevels=41, axis=None):

    if axis is None:
        axis = plt.gca()

    # --- load both ---
    profile1 = ProfileArraysDictionary_2D[parcelTypes[0]][parcelDepth][varName]['profile_array_right']
    count1   = ProfileArraysDictionary_2D[parcelTypes[0]][parcelDepth][varName]['profile_array_right_count']

    profile2 = ProfileArraysDictionary_2D[parcelTypes[1]][parcelDepth][varName]['profile_array_right']
    count2   = ProfileArraysDictionary_2D[parcelTypes[1]][parcelDepth][varName]['profile_array_right_count']

    mean1 = NanDivide(profile1, count1)
    mean2 = NanDivide(profile2, count2)

    if varName == "QCQI":
        mean1 = np.where(mean1 <= 1e-6, np.nan, mean1)
        mean2 = np.where(mean2 <= 1e-6, np.nan, mean2)

    times = np.arange(Nt) * ModelData.dt / 3600 + 6
    z = ModelData.zh

    multiplier = 1000 if "Q" in varName else 1

    data1 = multiplier * mean1.T
    data2 = multiplier * mean2.T

    # difference
    dataDiff = NanSubtract(data1, data2)

    # --- symmetric limits around 0 ---
    vmax = np.nanpercentile(np.abs(dataDiff), 95)
    vmin = -vmax

    nLevels = nLevels if nLevels % 2 == 1 else nLevels + 1
    levels = np.linspace(vmin, vmax, nLevels)

    if plotType == 'contour':
        plot = axis.contourf(times, z, dataDiff,
                             cmap='RdBu_r', levels=levels, extend='both')
    else:
        norm = mcolors.BoundaryNorm(levels, ncolors=plt.cm.RdBu_r.N, extend='both')
        plot = axis.pcolormesh(times, z, dataDiff,
                               cmap='RdBu_r', norm=norm, shading='auto')

    return plot

def MakeCombinedPlot_V1(parcelTypes, parcelDepth, plotType='contour', nLevels=41):

    varNames = list(cmapDictionary.keys())

    fig, axes = plt.subplots(
        2, len(varNames),
        figsize=(4 * len(varNames), 8),
        sharex=True,
        sharey=True,
        constrained_layout=True
    )

    for col, varName in enumerate(varNames):

        # --- Top row: CL ---
        ax_top = axes[0, col]
        im_top = MakePlot(parcelTypes[0], parcelDepth, varName,
                          plotType=plotType, nLevels=nLevels, axis=ax_top)
        ax_top.set_title(f"{parcelTypes[0]} - {varName}")

        # --- Bottom row: nonCL ---
        ax_bottom = axes[1, col]
        im_bottom = MakePlot(parcelTypes[1], parcelDepth, varName,
                             plotType=plotType, nLevels=nLevels, axis=ax_bottom)
        ax_bottom.set_title(f"{parcelTypes[1]} - {varName}")

        # Colorbars (one per column is cleaner)
        fig.colorbar(im_top, ax=[ax_top, ax_bottom], 
                     orientation='vertical',
                     aspect=30,pad=0.02)

    axes[0, 0].set_ylabel("z")
    axes[1, 0].set_ylabel("z")
    for ax in axes[1, :]:
        ax.set_xlabel("time (hr)")
        ax.set_xlim(left=10)
        ax.set_ylim(top=6)

    return fig

def MakeCombinedPlot_V2(parcelTypes, parcelDepth, plotType='contour', nLevels=41):

    varNames = list(cmapDictionary.keys())

    fig, axes = plt.subplots(
        3, len(varNames),   # now 3 rows
        figsize=(4 * len(varNames), 12),
        sharex=True,
        sharey=True,
        constrained_layout=True
    )

    for col, varName in enumerate(varNames):

        # --- row 1 ---
        ax_top = axes[0, col]
        im_top = MakePlot(parcelTypes[0], parcelDepth, varName,
                          plotType=plotType, nLevels=nLevels, axis=ax_top)
        ax_top.set_title(f"{parcelTypes[0]} - {varName}")

        # --- row 2 ---
        ax_mid = axes[1, col]
        im_mid = MakePlot(parcelTypes[1], parcelDepth, varName,
                          plotType=plotType, nLevels=nLevels, axis=ax_mid)
        ax_mid.set_title(f"{parcelTypes[1]} - {varName}")

        # --- row 3 (difference) ---
        ax_bot = axes[2, col]
        im_diff = MakeDifferencePlot(parcelTypes, parcelDepth, varName,
                                     plotType=plotType, nLevels=nLevels, axis=ax_bot)
        ax_bot.set_title(f"Δ ({parcelTypes[0]} - {parcelTypes[1]})")

        # --- colorbars ---
        fig.colorbar(im_top, ax=[ax_top, ax_mid],
                     orientation='vertical', aspect=30, pad=0.02)

        fig.colorbar(im_diff, ax=ax_bot,
                     orientation='vertical', aspect=30, pad=0.02)

    # labels
    axes[0, 0].set_ylabel("z")
    axes[1, 0].set_ylabel("z")
    axes[2, 0].set_ylabel("z")

    for ax in axes[2, :]:
        ax.set_xlabel("time (hr)")
        ax.set_xlim(left=10)
        ax.set_ylim(top=6)

    return fig

In [ ]:
def MakeMeanProfilePlot_V1(parcelType, parcelDepth, varName, 
                           linestyle,color,
                           axis=None):

    if axis is None:
        axis = plt.gca()

    profile = ProfileArraysDictionary_2D[parcelType][parcelDepth][varName]['profile_array_right']
    count   = ProfileArraysDictionary_2D[parcelType][parcelDepth][varName]['profile_array_right_count']

    z = ModelData.zh

    multiplier = 1000 if "Q" in varName else 1

    # Sum over time FIRST
    sum_profile = np.nansum(profile, axis=0)
    sum_count   = np.nansum(count, axis=0)

    # Then divide (safe division)
    mean_over_time = NanDivide(sum_profile, sum_count)

    data = multiplier * mean_over_time

    # --- Apply height cutoff ---
    targetZ = 6.0
    zIndex = np.abs(z - targetZ).argmin() + 1

    z_plot = z[:zIndex]
    x_plot = data[:zIndex]

    axis.plot(x_plot, z_plot, label=parcelType, linestyle=linestyle,color=color)

    axis.set_xlabel(varName)
    axis.set_ylabel("z")
    axis.set_ylim(0, z[zIndex-1])

    # --- x-limits with buffer ---
    xmin = np.nanmin(x_plot)
    xmax = np.nanmax(x_plot)
    dx = xmax - xmin
    buffer = 0.15 * dx if dx > 0 else 0.1
    axis.set_xlim(xmin - buffer, xmax + buffer)

    return x_plot

def MakeMeanProfilePlot_V2(parcelType, parcelDepth, varName, 
                           linestyle,color, 
                           axis=None):

    if axis is None:
        axis = plt.gca()

    profile = ProfileArraysDictionary_2D[parcelType][parcelDepth][varName]['profile_array_right']
    count = ProfileArraysDictionary_2D[parcelType][parcelDepth][varName]['profile_array_right_count']
    mean_profile = NanDivide(profile, count)

    z = ModelData.zh

    multiplier = 1000 if "Q" in varName else 1
    data = multiplier * mean_profile  # shape: (time, z)

    # average over time (axis=0)
    mean_over_time = np.nanmean(data, axis=0)

    # --- Apply height mask ---
    targetZ = 6.0
    zIndex = np.abs(ModelData.zh - targetZ).argmin() + 1
    z_plot = z[:zIndex]
    x_plot = mean_over_time[:zIndex]

    axis.plot(x_plot, z_plot, label=parcelType, linestyle=linestyle,color=color)

    axis.set_xlabel(varName)
    axis.set_ylabel("z")
    axis.set_ylim(0, 6)

    # --- Set x-limits based only on visible region ---
    xmin = np.nanmin(x_plot)
    xmax = np.nanmax(x_plot)
    buffer = 0.15 * (xmax - xmin)
    axis.set_xlim(xmin - buffer, xmax + buffer)

    return x_plot  # return data so combined function can unify limits

colorDictionary = {'ALL':'black',
                   'SHALLOW':'green',
                   'DEEP':'blue'}
def MakeCombinedMeanProfilePlot(parcelTypes, parcelDepth,
                                linestyles=['-','--']):

    varNames = list(cmapDictionary.keys())
    color=colorDictionary[parcelDepth]

    fig, axes = plt.subplots(
        1, len(varNames),
        figsize=(4 * len(varNames), 5),
        sharey=True
    )

    for col, varName in enumerate(varNames):
        ax = axes[col]

        for parcelType,linestyle in zip(parcelTypes,linestyles):
            MakeMeanProfilePlot_V1(parcelType, parcelDepth, varName,
                                   linestyle=linestyle,color=color, 
                                   axis=ax)

        ax.set_title(varName)
        ax.legend()

    axes[0].set_ylabel("z")

    plt.tight_layout()

    return fig

In [ ]:
#########################################
#PLOTTING

In [ ]:
parcelTypes = ['CL','nonCL']
parcelDepth = 'ALL'
fig=MakeCombinedPlot_V2(parcelTypes,parcelDepth)
fig=MakeCombinedMeanProfilePlot(parcelTypes,parcelDepth)

In [ ]:
parcelTypes = ['CL','nonCL']
parcelDepth = 'SHALLOW'
fig=MakeCombinedPlot_V2(parcelTypes,parcelDepth)
fig=MakeCombinedMeanProfilePlot(parcelTypes,parcelDepth)

In [ ]:
parcelTypes = ['CL','nonCL']
parcelDepth = 'DEEP'
fig=MakeCombinedPlot_V2(parcelTypes,parcelDepth)
fig=MakeCombinedMeanProfilePlot(parcelTypes,parcelDepth)